# End Goal
* This project compares the top 5 most streamed artist of 2024 (from a CSV file) with their current top 5 tracks in 2025 (through Spotify API). The main goal of this project is to analyze changes in popularity and streaming metrics through MySQL schema.



# Inserting CSV File, then connecting Locally to MySQL workbench and Spotify API

*   First, we are going to import our CSV file that we got online. From there we are going to clean the column names and any other data that needs cleaning.
*   Then from one of our labs we are going to copy the code we were given to we can connect to our mysql connector.
*   Lastly, we are going to use the code we were previously given in Lab 2 so that we can connect to our Spotify API.



In [ ]:
import pandas as pd
df = pd.read_csv("Most Streamed Spotify Songs 2024.csv", encoding='latin1')
df.head()


In [ ]:
#Cleaning Column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

#Converting date to datetime
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')

#Converting to actual nums
colsfix = [
    'spotify_streams', 'spotify_playlist_count', 'spotify_playlist_reach',
    'youtube_views', 'youtube_likes', 'tiktok_posts', 'tiktok_likes', 'tiktok_views',
    'youtube_playlist_reach', 'airplay_spins', 'siriusxm_spins',
    'deezer_playlist_reach', 'pandora_streams', 'pandora_track_stations',
    'soundcloud_streams', 'shazam_counts'
]

for col in colsfix:
    df[col] = df[col].replace(',', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

#Adding trackid
df['track_id'] = df.index + 1

#Checking to see if the data is clean and we can the correct columns and rows
print("Cleaned Dataset Shape:", df.shape)
df.head()

In [ ]:
import mysql.connector
import pandas as pd

mysql_address  = 'database-1.cd6cq8cy2yfo.us-east-2.rds.amazonaws.com'
mysql_username = 'admin'
mysql_password = 'DE_Student_PaSS'
mysql_database = 'my_dataengineering_db'

def get_conn_cur():
    cnx = mysql.connector.connect(
        user=mysql_username,
        password=mysql_password,
        host=mysql_address,
        database=mysql_database,
        port='3306'
    )
    return cnx, cnx.cursor()

def run_query(query_string):
    conn, cur = get_conn_cur()
    cur.execute(query_string)
    my_data = cur.fetchall()
    result_df = pd.DataFrame(my_data, columns=cur.column_names)
    cur.close()
    conn.close()
    return result_df

def sql_head(table_name):
    conn, cur = get_conn_cur()
    cur.execute(f"SELECT * FROM {table_name} LIMIT 5;")
    df = pd.DataFrame(cur.fetchall(), columns=cur.column_names)
    cur.close()
    conn.close()
    return df

In [ ]:
!pip install spotipy

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials


spotify_client_id = '9132f93fee5b4c31ac312a7fb9dc0c1c'
spotify_client_secret  = 'cf401566d4b0455495b7e743aeacdc5f'

client_credentials_manager = SpotifyClientCredentials(
    client_id=spotify_client_id,
    client_secret=spotify_client_secret
)

sp = spotipy.Spotify(
    client_credentials_manager=client_credentials_manager
)


# Getting Top 5 Artists and Current top tracks from Spotify API

In [ ]:
#Groups data by artist and returns top 5 based on spotify streams
top_artists_df = df.groupby('artist', as_index=False)['spotify_streams'].sum()
top_artists_df = top_artists_df.sort_values(by='spotify_streams', ascending=False).head(5)
top_artists_df

In [ ]:
#Gets top tracks for a given artist from the Spotify API and returns them as a DataFrame
def get_top_tracks_for_artist(artist_name, limit=5):
    results = sp.search(q=f"artist:{artist_name}", type="artist", limit=1)
    artist_items = results['artists']['items']
    if not artist_items:
        print(f"Artist '{artist_name}' not found on Spotify")
        return None
    artist_id = artist_items[0]['id']
    print(f"Artist: {artist_name}, ID: {artist_id}")

    top_tracks = sp.artist_top_tracks(artist_id, country='US')
    track_data = []
    for track in top_tracks['tracks'][:limit]:
        track_data.append({
            'artist_name': artist_name,
            'track_name': track['name'],
            'track_id': track['id'],
            'album_name': track['album']['name'],
            'release_date': track['album']['release_date'],
            'popularity': track['popularity'],
            'explicit': track['explicit']
        })
    return pd.DataFrame(track_data)

In [ ]:
#Goes through all 5 artists and gets their top 5 hits
top_artists_2024 = ["Bad Bunny", "The Weeknd", "Drake", "Taylor Swift", "Post Malone"]

all_tracks_df = pd.DataFrame()

for artist in top_artists_2024:
    artist_tracks = get_top_tracks_for_artist(artist, limit=5)
    if artist_tracks is not None:
        all_tracks_df = pd.concat([all_tracks_df, artist_tracks], ignore_index=True)

all_tracks_df

In [ ]:
#creating tables for artists, tracks and streaming metrics
drop_queries = [
    "DROP TABLE IF EXISTS StreamingMetrics;",
    "DROP TABLE IF EXISTS Tracks;",
    "DROP TABLE IF EXISTS Artists;"
]

create_queries = [
    """
    CREATE TABLE Artists (
        artist_id INT AUTO_INCREMENT PRIMARY KEY,
        artist_name VARCHAR(255) NOT NULL
    );
    """,
    """
    CREATE TABLE Tracks (
        track_id VARCHAR(50) PRIMARY KEY,
        track_name VARCHAR(255),
        album_name VARCHAR(255),
        release_date DATE,
        popularity INT,
        explicit BOOLEAN,
        artist_id INT,
        FOREIGN KEY (artist_id) REFERENCES Artists(artist_id)
    );
    """,
    """
    CREATE TABLE StreamingMetrics (
        metric_id INT AUTO_INCREMENT PRIMARY KEY,
        track_id VARCHAR(50),
        spotify_streams BIGINT,
        youtube_views BIGINT,
        tiktok_views BIGINT,
        airplay_spins INT,
        shazam_counts INT,
        FOREIGN KEY (track_id) REFERENCES Tracks(track_id)
    );
    """
]

conn, cur = get_conn_cur()

for query in drop_queries:
    cur.execute(query)

for query in create_queries:
    cur.execute(query)

conn.commit()
cur.close()
conn.close()

In [ ]:
#Cleans track and artists names, then merges the 2025 data with the 2024
df = df.rename(columns={'track': 'track_name'})

df['artist'] = df['artist'].str.strip()
df['track_name'] = df['track_name'].str.strip()
all_tracks_df['track_name'] = all_tracks_df['track_name'].str.strip()
all_tracks_df['artist_name'] = all_tracks_df['artist_name'].str.strip()

merged_df = pd.merge(
    all_tracks_df,
    df,
    how='left',
    left_on=['track_name', 'artist_name'],
    right_on=['track_name', 'artist']
)

merged_df.head()



In [ ]:
#These are for the next 3 cells get unique artists names, gets artist id, gets tracks based on artist id, then insertes each one into the database
def insert_artists(df):
    conn, cur = get_conn_cur()
    for artist in df['artist_name'].unique():
        cur.execute("""
            INSERT INTO Artists (artist_name)
            VALUES (%s) ON DUPLICATE KEY UPDATE artist_name = artist_name;
        """, (artist,))
    conn.commit()
    cur.close()
    conn.close()

In [ ]:
def get_artist_id_map():
    conn, cur = get_conn_cur()
    cur.execute("SELECT artist_id, artist_name FROM Artists;")
    rows = cur.fetchall()
    cur.close()
    conn.close()
    return {name: id for id, name in rows}

In [ ]:
def insert_tracks(df, artist_id_map):
    conn, cur = get_conn_cur()
    for _, row in df.iterrows():
        cur.execute("""
            INSERT INTO Tracks (
                track_id, track_name, album_name, release_date,
                popularity, explicit, artist_id
            ) VALUES (%s, %s, %s, %s, %s, %s, %s)
            ON DUPLICATE KEY UPDATE track_name = VALUES(track_name);
        """, (
            row['track_id'],
            row['track_name'],
            row['album_name'],
            row['release_date'],
            row['popularity'],
            row['explicit'],
            artist_id_map.get(row['artist_name'])
        ))
    conn.commit()
    cur.close()
    conn.close()


In [ ]:
insert_artists(all_tracks_df)
artist_id_map = get_artist_id_map()
insert_tracks(all_tracks_df, artist_id_map)

In [ ]:
sql_head("Tracks")

In [ ]:
#Cleans and converts the numeric streaming metrics then inserts them into the sql table
import numpy as np
import pandas as pd

numeric_cols = ['spotify_streams', 'youtube_views', 'tiktok_views', 'airplay_spins', 'shazam_counts']

for col in numeric_cols:
    metrics_df[col] = pd.to_numeric(metrics_df[col], errors='coerce')

for col in numeric_cols:
    metrics_df[col] = metrics_df[col].astype('Int64')

metrics_df = metrics_df.replace({pd.NA: None, np.nan: None})
conn, cur = get_conn_cur()

for _, row in metrics_df.iterrows():
    insert_query = """
    INSERT INTO StreamingMetrics (track_id, spotify_streams, youtube_views, tiktok_views, airplay_spins, shazam_counts)
    VALUES (%s, %s, %s, %s, %s, %s)
    """
    cur.execute(insert_query, (
        row['track_id'],
        row['spotify_streams'],
        row['youtube_views'],
        row['tiktok_views'],
        row['airplay_spins'],
        row['shazam_counts']
    ))

conn.commit()
cur.close()
conn.close()
sql_head("StreamingMetrics")

#Queries
The queries in the section will be random queries that check different things from the CSV file, and or the Spotify API

*   Total Number of tracks per artist (only top 5 popular)
*   Top 3 tracks on Spotify (based on streams)
*   Most recent track
*   Avg Popularity Score
*   Top 5 tracks on Shazam





In [ ]:
run_query('''
SELECT A.artist_name, COUNT(T.track_id) AS num_tracks
FROM Artists A
JOIN Tracks T ON A.artist_id = T.artist_id
GROUP BY A.artist_name
ORDER BY num_tracks DESC;
''')

In [ ]:
run_query('''
SELECT T.track_name, A.artist_name, S.spotify_streams
FROM StreamingMetrics S
JOIN Tracks T ON S.track_id = T.track_id
JOIN Artists A ON T.artist_id = A.artist_id
ORDER BY S.spotify_streams DESC
LIMIT 3;
''')

In [ ]:
run_query('''
SELECT A.artist_name, T.track_name, T.release_date
FROM Tracks T
JOIN Artists A ON T.artist_id = A.artist_id
WHERE (T.artist_id, T.release_date) IN (
    SELECT artist_id, MAX(release_date)
    FROM Tracks
    GROUP BY artist_id
);
''')

In [ ]:
run_query('''
SELECT A.artist_name, ROUND(AVG(T.popularity), 2) AS avg_popularity
FROM Artists A
JOIN Tracks T ON A.artist_id = T.artist_id
GROUP BY A.artist_name
ORDER BY avg_popularity DESC;
''')

In [ ]:
run_query('''
SELECT T.track_name, A.artist_name, S.shazam_counts
FROM StreamingMetrics S
JOIN Tracks T ON S.track_id = T.track_id
JOIN Artists A ON T.artist_id = A.artist_id
ORDER BY S.shazam_counts DESC
LIMIT 5;
''')

# Plots

* The plots in this section will just be representing what the queries did.



In [ ]:
import matplotlib.pyplot as plt

# Extract release year
merged_df['release_year'] = pd.to_datetime(merged_df['release_date_x']).dt.year

# Count tracks by artist and year
tracks_per_year = merged_df.groupby(['artist_name', 'release_year']).size().reset_index(name='count')

# Line plot
plt.figure(figsize=(9,6))
for artist in tracks_per_year['artist_name'].unique():
    subset = tracks_per_year[tracks_per_year['artist_name'] == artist]
    plt.plot(subset['release_year'], subset['count'], marker='o', label=artist)

plt.title("Number of Tracks per Year")
plt.xlabel("Year")
plt.ylabel("Number of Tracks")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Average popularity per artist per year
popularity_trend = merged_df.groupby(['artist_name', 'release_year'])['popularity'].mean().reset_index()

# Line plot
plt.figure(figsize=(9,6))
for artist in popularity_trend['artist_name'].unique():
    subset = popularity_trend[popularity_trend['artist_name'] == artist]
    plt.plot(subset['release_year'], subset['popularity'], marker='o', label=artist)

plt.title("Average Popularity 2014-2024")
plt.xlabel("Year")
plt.ylabel("Average Popularity")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9,6))
merged_df['spotify_streams'].dropna().astype(int).hist(bins=15, edgecolor='black')
plt.title("Spotify Streams")
plt.xlabel("Streams")
plt.ylabel("Frequent Streams")
plt.grid(True)
plt.show()